In [32]:
import pandas as pd
import os

In [ ]:
data_folder = '../data/'
all_dataframes = []

#create the good colum names
column_names = [
    'Link no', 'Name',
    'Start_LRP', 'Start_Offset', 'Start_Chainage',
    'End_LRP', 'End_Offset', 'End_Chainage',
    'Length_Km', 'Heavy_Truck', 'Medium_Truck', 'Small_Truck',
    'Large_Bus', 'Medium_Bus', 'Micro_Bus', 'Utility',
    'Car', 'Auto_Rickshaw', 'Motor_Cycle', 'Bi_Cycle',
    'Cycle_Rickshaw', 'Cart', 'Motorized', 'Non_Motorized', 'Total_AADT'
]

for filename in os.listdir(data_folder):
    if filename.endswith(".traffic.htm"):
        file_path = os.path.join(data_folder, filename)
        tables = pd.read_html(file_path, encoding='ISO-8859-1')
        if len(tables) > 0:
            #get the last table, and we only need the first 25 columns.
            df = tables[-1].iloc[:, :25]

            #skip the first three lines with html, and get the real data
            df = df.iloc[3:].copy()

            #make it the right column names
            df.columns = column_names

            #add the road
            df['Road'] = filename.split('.')[0]

            all_dataframes.append(df)

#put all together
final_df = pd.concat(all_dataframes, ignore_index=True)

#road is final column
cols = ['Road'] + [c for c in final_df.columns if c != 'Road']
final_df = final_df[cols]

# Toon het resultaat
display(final_df.head(30))

In [ ]:
display(final_df.tail())

In [ ]:
#check for example the N106
n106_data = final_df[final_df['Road'] == 'N106']
display(n106_data)

In [ ]:
final_df.columns

In [ ]:
df1 = pd.read_csv("../data/_roads3.csv")

In [ ]:
def merge_lrp_datasets(df_points: pd.DataFrame, df_links: pd.DataFrame) -> pd.DataFrame:
    """
    Merges LRP point data with link-level traffic data.

    For each point in df_points, finds which link range it falls into
    (based on chainage) and appends the average of Small_truck, Medium_truck,
    Heavy_truck from that link as a new column 'avg_truck_AADT'.

    Assumes:
    - df_points has columns: 'road', 'chainage'
    - df_links has columns: 'Road', 'Start_Chainage', 'End_Chainage',
                            'Small_truck', 'Medium_truck', 'Heavy_truck'
    """

    # Compute average truck AADT per link
    df_links = df_links.copy()
    df_links['Small_Truck'] = pd.to_numeric(df_links['Small_Truck'], errors='coerce')
    df_links['Medium_Truck'] = pd.to_numeric(df_links['Medium_Truck'], errors='coerce')
    df_links['Heavy_Truck'] = pd.to_numeric(df_links['Heavy_Truck'], errors='coerce')
    df_links['Start_Chainage'] = pd.to_numeric(df_links['Start_Chainage'], errors='coerce')
    df_links['End_Chainage'] = pd.to_numeric(df_links['End_Chainage'], errors='coerce')
    df_links['avg_truck_AADT'] = df_links[['Small_Truck', 'Medium_Truck', 'Heavy_Truck']].mean(axis=1)


    def get_avg_truck(row):
        road = row['road']
        chainage = row['chainage']

        # Filter links for the same road
        road_links = df_links[df_links['Road'] == road]

        # Find the link whose range contains this chainage
        # A point belongs to a link if: Start_Chainage <= chainage <= End_Chainage
        match = road_links[
            (road_links['Start_Chainage'] <= chainage) &
            (road_links['End_Chainage'] >= chainage)
        ]

        if not match.empty:
            return match.iloc[0]['avg_truck_AADT']

        # Edge case: chainage is beyond all link ranges (e.g. very last point)
        # Assign to the last link of the road
        if not road_links.empty and chainage > road_links['End_Chainage'].max():
            return road_links.loc[road_links['End_Chainage'].idxmax(), 'avg_truck_AADT']

        return None  # No matching link found

    df_points = df_points.copy()
    df_points['avg_truck_AADT'] = df_points.apply(get_avg_truck, axis=1)

    return df_points



In [ ]:
df_result = merge_lrp_datasets(df1, final_df)

In [ ]:
df_result.head(30)

In [ ]:
df_result.to_csv('../data/data_with_AADT.csv')